# Plot spot position as time series data
env : data_vis_32

# 1.0 Import relevant packages

In [8]:
# import pypyodbc
import pandas as pd
import plotly.express as px
from matplotlib.colors import to_hex
import seaborn as sns
import re
from pathlib import Path

# 2.0 Import spot position and size QA data

In [9]:
data_path = r"../data/xlsx_exported_from_access/SpotPositionResults.xlsx"

df = pd.read_excel(data_path)

df.head(2)



,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,hor_rt_gradient,hor_lt_gradient,hor_fwhm,vert_rt_gradient,vert_lt_gradient,vert_fwhm,bltr_rt_gradient,bltr_lt_gradient,bltr_fwhm,tlbr_rt_gradient,tlbr_lt_gradient,tlbr_fwhm
0,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Centre,-0.2357,124.9224,-9.059840,9.157258,13.342140,-8.964427,9.333333,13.536155,-8.485281,8.747554,14.216962,-8.909545,8.992812,13.842831
1,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Left,-125.0153,125.4501,-8.871094,9.120482,13.562307,-8.964427,9.333333,13.712522,-8.747554,8.591347,14.310494,-8.747554,8.992812,13.842831


# 3.0 exploratory data analysis - understand your data

In [10]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41172 entries, 0 to 41171
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ADate             41172 non-null  datetime64[ns]
 1   MachineName       41172 non-null  object        
 2   Energy            41172 non-null  int64         
 3   Device            41172 non-null  object        
 4   Gantry Angle      41172 non-null  int64         
 5   Spot              41172 non-null  object        
 6   x-pos             41172 non-null  float64       
 7   y-pos             41172 non-null  float64       
 8   hor_rt_gradient   41172 non-null  float64       
 9   hor_lt_gradient   41172 non-null  float64       
 10  hor_fwhm          41172 non-null  float64       
 11  vert_rt_gradient  41172 non-null  float64       
 12  vert_lt_gradient  41172 non-null  float64       
 13  vert_fwhm         41172 non-null  float64       
 14  bltr_rt_gradient  4117

In [11]:
df.value_counts("MachineName"), df.value_counts("Device"), df.value_counts("Energy")

(MachineName
 Gantry 3    10576
 Gantry 1    10472
 Gantry 4    10357
 Gantry 2     9767
 Name: count, dtype: int64,
 Device
 XRV-3000    31815
 XRV-4000     9357
 Name: count, dtype: int64,
 Energy
 150    8250
 240    8243
 200    8233
 100    8232
 70     8214
 Name: count, dtype: int64)

# 4.0 filtering data

In [12]:
sub_df = df[["ADate",	"MachineName", 	"Energy", "Device", "Gantry Angle", "Spot", "x-pos", "y-pos"]].copy()

## calculate abs shift

In [13]:
pred_xrv4000 = {'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175], \
                'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125], \
                'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]}

sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

sub_df['abs_xpos'] = sub_df["x-pos"] - sub_df["px_pos"]
sub_df['abs_ypos'] = sub_df["y-pos"] - sub_df["py_pos"]

In [14]:
print(sub_df.head(2))

                ADate MachineName  Energy    Device  Gantry Angle  \
0 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   
1 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   

            Spot     x-pos     y-pos  px_pos  py_pos  abs_xpos  abs_ypos  
0  Bottom-Centre   -0.2357  124.9224       0     125   -0.2357   -0.0776  
1    Bottom-Left -125.0153  125.4501    -125     125   -0.0153    0.4501  


In [15]:

def plotly_spot_position(df, pos, gantry, device, energy, gantry_angle, n_months):
    """ plot spot position time series data
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        pos = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int,
        gantry_angle = 0,90,180,270
        n_month = int

    
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]

    # set colour
    palette = sns.color_palette("deep", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]


    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=pos,
        symbol='Spot', 
        color='Spot',        # hue
         color_discrete_sequence= px.colors.qualitative.T10,
        title=f'{gantry} - absolute shift- {pos}',
        labels={'x-pos': 'X Position', 'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")



    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))

    # Show plot
    fig.show()

    
    return 


# plotting absolute y-pos, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last2 months
plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)


In [16]:
plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)

## plot another device

In [17]:
plotly_spot_position(sub_df, "abs_xpos", "Gantry 4", "XRV-3000", 70, 0, 24)

In [18]:
start_date = pd.Timestamp.today() - pd.DateOffset(months=12)
selected_df = sub_df[(df["MachineName"]=="Gantry 2") & (df["Device"] == "XRV-3000") & (df['ADate'] >= start_date)].copy()
# Calculate average abs_xpos per adate and energy
selected_df['avg_abs_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])["abs_xpos"].transform('mean')

selected_df.head(5)

,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,px_pos,py_pos,abs_xpos,abs_ypos,avg_abs_pos
36723,2025-09-09 20:29:00,Gantry 2,70,XRV-3000,0,Bottom-Centre,0.0924,125.5803,0,125,0.0924,0.5803,0.185778
36724,2025-09-09 20:29:00,Gantry 2,70,XRV-3000,0,Bottom-Left,-124.9983,125.3955,-125,125,0.0017,0.3955,0.185778
36725,2025-09-09 20:29:00,Gantry 2,70,XRV-3000,0,Bottom-Right,125.3825,125.6210,125,125,0.3825,0.6210,0.185778
36726,2025-09-09 20:29:00,Gantry 2,70,XRV-3000,0,Centre,0.1051,0.5249,0,0,0.1051,0.5249,0.185778
36727,2025-09-09 20:29:00,Gantry 2,70,XRV-3000,0,Left,-124.9959,0.8485,-125,0,0.0041,0.8485,0.185778


In [19]:

def plotly_ave_spot_position(df, parameter, gantry, device,  n_months):
    """ plot average spot position across all spot positions with the same adate and energy
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        paramter = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int
        n_month = int

    
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date)].copy()


    # Calculate average abs_xpos per adate and energy
    selected_df['avg_abs_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])[parameter].transform('mean')

    # want to displace energy as discrete colour not spectrum
    selected_df['Energy'] = df['Energy'].astype(int).astype(str)

    

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y='avg_abs_pos',
        symbol='Gantry Angle', 
        color='Energy',        # hue
        title=f'average {parameter} across all spot positions with the same adate and energy',
        labels={'x-pos': 'X Position', 'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")



    # Optional: connect points by spot for clarity
       # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))

    # Show plot
    fig.show()

    
    return 




In [20]:
plotly_ave_spot_position(sub_df, "abs_xpos", "Gantry 1", "XRV-3000",  24)

# FWHM

## Read ref data

In [21]:
from pathlib import Path

ref_path = Path(data_path).resolve().parent / "ref" / "Ref_RS0_Dist0.xlsx"
ref_df = pd.read_excel(ref_path)

print(ref_df.columns)
print(ref_df.head())

print(ref_df.columns)
print(ref_df.head())

Index(['source', 'gantry', 'rs', 'dist', 'energy', 'x_stddev', 'y_stddev'], dtype='object')
  source  gantry  rs  dist  energy  x_stddev  y_stddev
0     G3     270   0     0      70  5.913573  6.000366
1     G3     270   0     0      75  5.808925  5.796172
2     G3     270   0     0      80  5.605678  5.515724
3     G3     270   0     0      85  5.494827  5.322688
4     G3     270   0     0      90  5.389959  5.203662
Index(['source', 'gantry', 'rs', 'dist', 'energy', 'x_stddev', 'y_stddev'], dtype='object')
  source  gantry  rs  dist  energy  x_stddev  y_stddev
0     G3     270   0     0      70  5.913573  6.000366
1     G3     270   0     0      75  5.808925  5.796172
2     G3     270   0     0      80  5.605678  5.515724
3     G3     270   0     0      85  5.494827  5.322688
4     G3     270   0     0      90  5.389959  5.203662


## Read data + generate average

In [22]:
fwhm_base_cols = ["hor_fwhm", "vert_fwhm", "bltr_fwhm", "tlbr_fwhm"]

fwhm_df = df[[
    "ADate", "MachineName", "Energy", "Device", "Gantry Angle", "Spot", *fwhm_base_cols
]].copy()

fwhm_df["ave_fwhm"] = fwhm_df[fwhm_base_cols].mean(axis=1)

# define the full list
fwhm_cols = fwhm_base_cols + ["ave_fwhm"]

FWHM_FACTOR = 2.3548200450309493

In [23]:
print(fwhm_df.columns.tolist())

['ADate', 'MachineName', 'Energy', 'Device', 'Gantry Angle', 'Spot', 'hor_fwhm', 'vert_fwhm', 'bltr_fwhm', 'tlbr_fwhm', 'ave_fwhm']


In [24]:
def get_hls_palette_hex(n_colors):
    return [to_hex(c) for c in sns.color_palette("hls", n_colors=n_colors)]

## ----- Plot 1: 4 directions

In [25]:
def plotly_fwhm_time_series(
    df,
    gantry,
    device,
    energy,
    gantry_angle,
    n_months=12,
    fwhm_col="hor_fwhm",
    ref_df=None,
    ref_source="TPS",
    tol_frac=0.10
):
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    base_cols = ["hor_fwhm", "vert_fwhm", "bltr_fwhm", "tlbr_fwhm"]
    allowed_single = base_cols + ["ave_fwhm"]

    sel = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["Energy"] == energy) &
        (df["Gantry Angle"] == gantry_angle) &
        (df["ADate"] >= start_date)
    ].copy()

    if sel.empty:
        print("No data after filtering.")
        return

    sel["ave_fwhm"] = sel[base_cols].mean(axis=1)

    n_spots = sel["Spot"].nunique()
    color_seq = get_hls_palette_hex(n_spots)

    # title text with tolerance info
    if ref_df is not None:
        tol_title = f"<br><sup>Tolerance: {ref_source} FWHM ±{tol_frac:.0%}</sup>"
    else:
        tol_title = ""

    # ---- build plot ----
    if fwhm_col == "all":
        long_df = sel.melt(
            id_vars=["ADate", "Spot"],
            value_vars=base_cols,
            var_name="Direction",
            value_name="FWHM"
        )

        fig = px.scatter(
            long_df,
            x="ADate",
            y="FWHM",
            color="Spot",
            symbol="Spot",
            facet_col="Direction",
            facet_col_wrap=2,
            category_orders={"Direction": base_cols},
            color_discrete_sequence=color_seq,
            title=f"{gantry} {device} {energy}MeV GA={gantry_angle} — FWHM (all directions){tol_title}",
            height=700
        )
        fig.update_yaxes(matches=None)

    else:
        if fwhm_col not in allowed_single:
            raise ValueError(f"fwhm_col must be one of {allowed_single} or 'all'")

        fig = px.scatter(
            sel,
            x="ADate",
            y=fwhm_col,
            color="Spot",
            symbol="Spot",
            color_discrete_sequence=color_seq,
            title=f"{gantry} {device} {energy}MeV GA={gantry_angle} — {fwhm_col}{tol_title}",
            height=550
        )

    # ---- add tolerance from ref ----
    if ref_df is not None:
        if str(ref_source).strip().upper() == "TPS":
            ref_src = "G1"
            ref_ga = 90
        else:
            ref_src = ref_source
            ref_ga = gantry_angle

        ref = ref_df[
            (ref_df["source"].astype(str).str.strip().str.upper() == str(ref_src).strip().upper()) &
            (pd.to_numeric(ref_df["gantry"], errors="coerce") == ref_ga) &
            (pd.to_numeric(ref_df["energy"], errors="coerce") == energy)
        ]

        if not ref.empty:
            r = ref.iloc[0]
            x_sigma = float(r["x_stddev"])
            y_sigma = float(r["y_stddev"])

            hor_center = x_sigma * FWHM_FACTOR
            vert_center = y_sigma * FWHM_FACTOR
            ave_center = (hor_center + vert_center) / 2.0

            def add_tol(center, row=1, col=1):
                fig.add_hline(y=center * (1 + tol_frac), line_dash="dash", line_color="grey", line_width=2, row=row, col=col)
                fig.add_hline(y=center * (1 - tol_frac), line_dash="dash", line_color="grey", line_width=2, row=row, col=col)

            if fwhm_col == "hor_fwhm":
                add_tol(hor_center)
            elif fwhm_col == "vert_fwhm":
                add_tol(vert_center)
            elif fwhm_col == "ave_fwhm":
                add_tol(ave_center)
            elif fwhm_col == "all":
                add_tol(hor_center, row=2, col=1)
                add_tol(vert_center, row=2, col=2)

    fig.update_traces(
        mode='markers+lines',
        marker=dict(size=12, line=dict(width=2), opacity=0.65),
        line=dict(width=1)
    )

    fig.show()

In [26]:
plotly_fwhm_time_series(
    fwhm_df,
    gantry="Gantry 4",
    device="XRV-4000",
    energy=100,
    gantry_angle=0,
    n_months=24,
    fwhm_col="all",
    ref_df=ref_df,
    ref_source="TPS",
    tol_frac=0.10
)


## ----- Plot 2: Average
Same function just plotting the average

In [27]:
plotly_fwhm_time_series(
    fwhm_df,
    gantry="Gantry 4",
    device="XRV-4000",
    energy=100,
    gantry_angle=0,
    n_months=24,
    fwhm_col="ave_fwhm",
    ref_df=ref_df,
    ref_source="TPS",
    tol_frac=0.10
)

## Matrix: with energy selectors

In [28]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plotly_fwhm_spot_matrix(
    df,
    gantry,
    device,
    energy=None,          # None / single value / list of energies
    gantry_angle=0,
    n_months=12,
    fwhm_col="hor_fwhm",  # "hor_fwhm" / "vert_fwhm" / "bltr_fwhm" / "tlbr_fwhm" / "ave_fwhm"
    ref_df=None,
    ref_source="TPS",
    tol_frac=0.10
):
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    base_cols = ["hor_fwhm", "vert_fwhm", "bltr_fwhm", "tlbr_fwhm"]
    allowed_single = base_cols + ["ave_fwhm"]

    if fwhm_col not in allowed_single:
        raise ValueError(f"fwhm_col must be one of {allowed_single}")

    sel = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["Gantry Angle"] == gantry_angle) &
        (df["ADate"] >= start_date)
    ].copy()

    if energy is not None:
        if not isinstance(energy, (list, tuple, set)):
            energy = [energy]
        sel = sel[sel["Energy"].isin(energy)].copy()

    if sel.empty:
        print("No data after filtering.")
        return

    sel["ave_fwhm"] = sel[base_cols].mean(axis=1)

    energies = sorted(sel["Energy"].dropna().unique())
    color_seq = get_hls_palette_hex(len(energies))

    # Physical layout should be this
    spot_grid = [
        ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right"],
        ["Top-Left", "Top-Centre", "Top-Right"],
        ["Left", "Centre", "Right"],
        ["Bottom-Left", "Bottom-Centre", "Bottom-Right"],
        ["Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]
    ]

    n_rows = len(spot_grid)
    n_cols = len(spot_grid[0])

    spot_order = [s for row in spot_grid for s in row]
    spot_pos = {
        spot_grid[r][c]: (r + 1, c + 1)
        for r in range(n_rows)
        for c in range(n_cols)
    }

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=spot_order,
        shared_xaxes=True,
        shared_yaxes=True,
        vertical_spacing=0.04,
        horizontal_spacing=0.05
    )

    legend_done = set()

    # ref selection rule
    if str(ref_source).strip().upper() == "TPS":
        ref_src = "G1"
        ref_ga = 90
    else:
        ref_src = ref_source
        ref_ga = gantry_angle

    x_min = sel["ADate"].min()
    x_max = sel["ADate"].max()

    for e, color in zip(energies, color_seq):
        sub_e = sel[sel["Energy"] == e].copy()
        legend_group = f"{e}"

        # ---- get tolerance centre for this energy ----
        tol_center = None
        if ref_df is not None:
            ref = ref_df[
                (ref_df["source"].astype(str).str.strip().str.upper() == str(ref_src).strip().upper()) &
                (pd.to_numeric(ref_df["gantry"], errors="coerce") == ref_ga) &
                (pd.to_numeric(ref_df["energy"], errors="coerce") == e)
            ]

            if not ref.empty:
                r = ref.iloc[0]
                x_sigma = float(r["x_stddev"])
                y_sigma = float(r["y_stddev"])

                hor_center = x_sigma * FWHM_FACTOR
                vert_center = y_sigma * FWHM_FACTOR
                ave_center = (hor_center + vert_center) / 2.0

                if fwhm_col == "hor_fwhm":
                    tol_center = hor_center
                elif fwhm_col == "vert_fwhm":
                    tol_center = vert_center
                else:
                    # for ave_fwhm and diagonal directions, use average ref
                    tol_center = ave_center

        # ---- add traces for each spot ----
        for spot in spot_order:
            row, col = spot_pos[spot]
            sub_spot = sub_e[sub_e["Spot"] == spot].sort_values("ADate")

            # add tolerance as traces, not shapes, so legend can control them
            if tol_center is not None:
                y_hi = tol_center * (1 + tol_frac)
                y_lo = tol_center * (1 - tol_frac)

                fig.add_trace(
                    go.Scatter(
                        x=[x_min, x_max],
                        y=[y_hi, y_hi],
                        mode="lines",
                        line=dict(color=color, width=1, dash="dash"),
                        legendgroup=legend_group,
                        showlegend=False,
                        hoverinfo="skip"
                    ),
                    row=row,
                    col=col
                )

                fig.add_trace(
                    go.Scatter(
                        x=[x_min, x_max],
                        y=[y_lo, y_lo],
                        mode="lines",
                        line=dict(color=color, width=1, dash="dash"),
                        legendgroup=legend_group,
                        showlegend=False,
                        hoverinfo="skip"
                    ),
                    row=row,
                    col=col
                )

            if sub_spot.empty:
                continue

            showlegend = e not in legend_done

            fig.add_trace(
                go.Scatter(
                    x=sub_spot["ADate"],
                    y=sub_spot[fwhm_col],
                    mode="lines+markers",
                    name=f"{e} MeV",
                    legendgroup=legend_group,
                    showlegend=showlegend,
                    line=dict(color=color, width=1.6),
                    marker=dict(
                        symbol="circle",   # dot
                        size=5,
                        color=color
                    ),
                    hovertemplate=(
                        f"Spot={spot}<br>"
                        f"Energy={e} MeV<br>"
                        "Date=%{x|%Y-%m-%d}<br>"
                        f"{fwhm_col}=%{{y:.2f}}<extra></extra>"
                    )
                ),
                row=row,
                col=col
            )

            legend_done.add(e)

    fig.update_layout(
        height=1200,
        width=1050,
        template="plotly_white",
        title=f"{gantry} {device} GA={gantry_angle} — {fwhm_col} by spot location",
        legend_title_text="Energy",
        legend=dict(
            groupclick="togglegroup"
        )
    )

    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=True)

    fig.show()

In [29]:
plotly_fwhm_spot_matrix(
    fwhm_df,
    gantry="Gantry 4",
    device="XRV-4000",
    n_months=24,
    energy=[70, 100, 150, 200, 240],
    gantry_angle=0,
    fwhm_col="ave_fwhm",
    ref_df=ref_df,
    ref_source="TPS"
)

## Matrix: with energy and GA selectors

In [30]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plotly_fwhm_spot_matrix(
    df,
    gantry,
    device,
    energy=None,              # None / single value / list
    gantry_angle=None,        # None / single value / list
    n_months=12,
    fwhm_col="hor_fwhm",      # "hor_fwhm" / "vert_fwhm" / "bltr_fwhm" / "tlbr_fwhm" / "ave_fwhm"
    ref_df=None,
    ref_source="TPS",
    tol_frac=0.10
):
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    base_cols = ["hor_fwhm", "vert_fwhm", "bltr_fwhm", "tlbr_fwhm"]
    allowed_single = base_cols + ["ave_fwhm"]

    if fwhm_col not in allowed_single:
        raise ValueError(f"fwhm_col must be one of {allowed_single}")

    sel = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date)
    ].copy()

    if energy is not None:
        if not isinstance(energy, (list, tuple, set)):
            energy = [energy]
        sel = sel[sel["Energy"].isin(energy)].copy()

    if gantry_angle is not None:
        if not isinstance(gantry_angle, (list, tuple, set)):
            gantry_angle = [gantry_angle]
        sel = sel[sel["Gantry Angle"].isin(gantry_angle)].copy()

    if sel.empty:
        print("No data after filtering.")
        return

    sel["Spot"] = sel["Spot"].astype(str).str.strip()
    sel["ave_fwhm"] = sel[base_cols].mean(axis=1)

    energies = sorted(sel["Energy"].dropna().unique())
    gantry_angles = sorted(sel["Gantry Angle"].dropna().unique())

    # one colour per energy
    energy_colors = dict(zip(energies, get_hls_palette_hex(len(energies))))

    # one marker per gantry angle
    ga_symbols = {
        0: "circle",
        90: "square",
        180: "diamond",
        270: "x",
    }

    # fallback if some unexpected angle appears
    fallback_symbols = ["circle", "square", "diamond", "x", "triangle-up", "triangle-down", "cross"]
    for i, ga in enumerate(gantry_angles):
        if ga not in ga_symbols:
            ga_symbols[ga] = fallback_symbols[i % len(fallback_symbols)]

    spot_grid = [
        ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right"],
        ["Top-Left", "Top-Centre", "Top-Right"],
        ["Left", "Centre", "Right"],
        ["Bottom-Left", "Bottom-Centre", "Bottom-Right"],
        ["Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]
    ]

    n_rows = len(spot_grid)
    n_cols = len(spot_grid[0])

    spot_order = [s for row in spot_grid for s in row]
    spot_pos = {
        spot_grid[r][c]: (r + 1, c + 1)
        for r in range(n_rows)
        for c in range(n_cols)
    }

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=spot_order,
        shared_xaxes=True,
        shared_yaxes=True,
        vertical_spacing=0.04,
        horizontal_spacing=0.05
    )

    legend_done = set()
    x_min = sel["ADate"].min()
    x_max = sel["ADate"].max()

    for e in energies:
        for ga in gantry_angles:
            sub = sel[
                (sel["Energy"] == e) &
                (sel["Gantry Angle"] == ga)
            ].copy()

            if sub.empty:
                continue

            color = energy_colors[e]
            symbol = ga_symbols[ga]
            combo_key = (e, ga)
            legend_group = f"E{e}_GA{ga}"
            legend_name = f"{e} MeV | GA {ga}"

            # ---- tolerance for this energy + gantry angle ----
            tol_center = None
            if ref_df is not None:
                if str(ref_source).strip().upper() == "TPS":
                    ref_src = "G1"
                    ref_ga = 90
                else:
                    ref_src = ref_source
                    ref_ga = ga

                ref = ref_df[
                    (ref_df["source"].astype(str).str.strip().str.upper() == str(ref_src).strip().upper()) &
                    (pd.to_numeric(ref_df["gantry"], errors="coerce") == ref_ga) &
                    (pd.to_numeric(ref_df["energy"], errors="coerce") == e)
                ]

                if not ref.empty:
                    r = ref.iloc[0]
                    x_sigma = float(r["x_stddev"])
                    y_sigma = float(r["y_stddev"])

                    hor_center = x_sigma * FWHM_FACTOR
                    vert_center = y_sigma * FWHM_FACTOR
                    ave_center = (hor_center + vert_center) / 2.0

                    if fwhm_col == "hor_fwhm":
                        tol_center = hor_center
                    elif fwhm_col == "vert_fwhm":
                        tol_center = vert_center
                    else:
                        tol_center = ave_center

            for spot in spot_order:
                row, col = spot_pos[spot]
                sub_spot = sub[sub["Spot"] == spot].sort_values("ADate")

                # tolerance traces
                if tol_center is not None:
                    y_hi = tol_center * (1 + tol_frac)
                    y_lo = tol_center * (1 - tol_frac)

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_hi, y_hi],
                            mode="lines",
                            line=dict(color=color, width=1, dash="dash"),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip"
                        ),
                        row=row,
                        col=col
                    )

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_lo, y_lo],
                            mode="lines",
                            line=dict(color=color, width=1, dash="dash"),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip"
                        ),
                        row=row,
                        col=col
                    )

                if sub_spot.empty:
                    continue

                showlegend = combo_key not in legend_done

                fig.add_trace(
                    go.Scatter(
                        x=sub_spot["ADate"],
                        y=sub_spot[fwhm_col],
                        mode="lines+markers",
                        name=legend_name,
                        legendgroup=legend_group,
                        showlegend=showlegend,
                        line=dict(color=color, width=1.5),
                        marker=dict(
                            symbol=symbol,
                            size=5,
                            color=color
                        ),
                        hovertemplate=(
                            f"Spot={spot}<br>"
                            f"Energy={e} MeV<br>"
                            f"GA={ga}<br>"
                            "Date=%{x|%Y-%m-%d}<br>"
                            f"{fwhm_col}=%{{y:.2f}}<extra></extra>"
                        )
                    ),
                    row=row,
                    col=col
                )

                legend_done.add(combo_key)

    fig.update_layout(
        height=1200,
        width=1050,
        template="plotly_white",
        title=f"{gantry} {device} — {fwhm_col} by spot location",
        legend_title_text="Energy | Gantry Angle",
        legend=dict(groupclick="togglegroup")
    )

    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=True)

    fig.show()

In [31]:
plotly_fwhm_spot_matrix(
    fwhm_df,
    gantry="Gantry 1",
    device="XRV-4000",
    energy=[70, 100, 150, 200, 240],
    gantry_angle=[0, 90, 180, 270],
    n_months=6,
    fwhm_col="ave_fwhm",
    ref_df=ref_df,
    ref_source="TPS"
)

No data after filtering.


## Matrix: with energy and GA selectors + bigger markers

In [32]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plotly_fwhm_spot_matrix(
    df,
    gantry,
    device,
    energy=None,              # None / single value / list
    gantry_angle=None,        # None / single value / list
    n_months=12,
    fwhm_col="hor_fwhm",      # "hor_fwhm" / "vert_fwhm" / "bltr_fwhm" / "tlbr_fwhm" / "ave_fwhm"
    ref_df=None,
    ref_source="TPS",
    tol_frac=0.10
):
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    base_cols = ["hor_fwhm", "vert_fwhm", "bltr_fwhm", "tlbr_fwhm"]
    allowed_single = base_cols + ["ave_fwhm"]

    if fwhm_col not in allowed_single:
        raise ValueError(f"fwhm_col must be one of {allowed_single}")

    sel = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date)
    ].copy()

    if energy is not None:
        if not isinstance(energy, (list, tuple, set)):
            energy = [energy]
        sel = sel[sel["Energy"].isin(energy)].copy()

    if gantry_angle is not None:
        if not isinstance(gantry_angle, (list, tuple, set)):
            gantry_angle = [gantry_angle]
        sel = sel[sel["Gantry Angle"].isin(gantry_angle)].copy()

    if sel.empty:
        print("No data after filtering.")
        return

    sel["Spot"] = sel["Spot"].astype(str).str.strip()
    sel["ave_fwhm"] = sel[base_cols].mean(axis=1)

    energies = sorted(sel["Energy"].dropna().unique())
    gantry_angles = sorted(sel["Gantry Angle"].dropna().unique())

    energy_colors = dict(zip(energies, get_hls_palette_hex(len(energies))))

    ga_symbols = {
        0: "circle",
        90: "square",
        180: "diamond",
        270: "x",
    }
    fallback_symbols = ["circle", "square", "diamond", "x", "triangle-up", "triangle-down", "cross"]
    for i, ga in enumerate(gantry_angles):
        if ga not in ga_symbols:
            ga_symbols[ga] = fallback_symbols[i % len(fallback_symbols)]

    # device-specific physical layout
    if str(device).strip().upper() == "XRV-3000":
        spot_grid = [
            ["Top-Left", "Top-Centre", "Top-Right"],
            ["Left", "Centre", "Right"],
            ["Bottom-Left", "Bottom-Centre", "Bottom-Right"]
        ]
    else:
        spot_grid = [
            ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right"],
            ["Top-Left", "Top-Centre", "Top-Right"],
            ["Left", "Centre", "Right"],
            ["Bottom-Left", "Bottom-Centre", "Bottom-Right"],
            ["Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]
        ]

    n_rows = len(spot_grid)
    n_cols = len(spot_grid[0])

    spot_order = [s for row in spot_grid for s in row]
    spot_pos = {
        spot_grid[r][c]: (r + 1, c + 1)
        for r in range(n_rows)
        for c in range(n_cols)
    }

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=spot_order,
        shared_xaxes=True,
        shared_yaxes=True,
        vertical_spacing=0.06 if n_rows == 3 else 0.04,
        horizontal_spacing=0.05
    )

    legend_done = set()
    x_min = sel["ADate"].min()
    x_max = sel["ADate"].max()

    for e in energies:
        for ga in gantry_angles:
            sub = sel[
                (sel["Energy"] == e) &
                (sel["Gantry Angle"] == ga)
            ].copy()

            if sub.empty:
                continue

            color = energy_colors[e]
            symbol = ga_symbols[ga]
            combo_key = (e, ga)
            legend_group = f"E{e}_GA{ga}"
            legend_name = f"{e} MeV | GA {ga}"

            tol_center = None
            if ref_df is not None:
                if str(ref_source).strip().upper() == "TPS":
                    ref_src = "G1"
                    ref_ga = 90
                else:
                    ref_src = ref_source
                    ref_ga = ga

                ref = ref_df[
                    (ref_df["source"].astype(str).str.strip().str.upper() == str(ref_src).strip().upper()) &
                    (pd.to_numeric(ref_df["gantry"], errors="coerce") == ref_ga) &
                    (pd.to_numeric(ref_df["energy"], errors="coerce") == e)
                ]

                if not ref.empty:
                    r = ref.iloc[0]
                    x_sigma = float(r["x_stddev"])
                    y_sigma = float(r["y_stddev"])

                    hor_center = x_sigma * FWHM_FACTOR
                    vert_center = y_sigma * FWHM_FACTOR
                    ave_center = (hor_center + vert_center) / 2.0

                    if fwhm_col == "hor_fwhm":
                        tol_center = hor_center
                    elif fwhm_col == "vert_fwhm":
                        tol_center = vert_center
                    else:
                        tol_center = ave_center

            for spot in spot_order:
                row, col = spot_pos[spot]
                sub_spot = sub[sub["Spot"] == spot].sort_values("ADate")

                if tol_center is not None:
                    y_hi = tol_center * (1 + tol_frac)
                    y_lo = tol_center * (1 - tol_frac)

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_hi, y_hi],
                            mode="lines",
                            line=dict(color=color, width=1, dash="dash"),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip"
                        ),
                        row=row,
                        col=col
                    )

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_lo, y_lo],
                            mode="lines",
                            line=dict(color=color, width=1, dash="dash"),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip"
                        ),
                        row=row,
                        col=col
                    )

                if sub_spot.empty:
                    continue

                showlegend = combo_key not in legend_done

                fig.add_trace(
                    go.Scatter(
                        x=sub_spot["ADate"],
                        y=sub_spot[fwhm_col],
                        mode="lines+markers",
                        name=legend_name,
                        legendgroup=legend_group,
                        showlegend=showlegend,
                        line=dict(color=color, width=1),
                        marker=dict(
                            symbol=symbol,
                            size=12,
                            color=color,
                            opacity=0.65,
                            line=dict(width=2)
                        ),
                        hovertemplate=(
                            f"Spot={spot}<br>"
                            f"Energy={e} MeV<br>"
                            f"GA={ga}<br>"
                            "Date=%{x|%Y-%m-%d}<br>"
                            f"{fwhm_col}=%{{y:.2f}}<extra></extra>"
                        )
                    ),
                    row=row,
                    col=col
                )

                legend_done.add(combo_key)

    fig.update_layout(
        height=850 if n_rows == 3 else 1200,
        width=1050,
        template="plotly_white",
        title=f"{gantry} {device} — {fwhm_col} by spot location",
        legend_title_text="Energy | Gantry Angle",
        legend=dict(groupclick="togglegroup")
    )

    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=True)

    fig.show()

In this version if its 3000 then there won't be any top-top or bottom-bottom data rows. 

In [33]:
plotly_fwhm_spot_matrix(
    fwhm_df,
    gantry="Gantry 1",
    device="XRV-3000",
    energy=[70, 100, 150, 200, 240],
    gantry_angle=[0, 90, 180, 270],
    n_months=12,
    fwhm_col="ave_fwhm",
    ref_df=ref_df,
    ref_source="TPS"
)

## ----- plot No. 3: Matrix with dropdowns

- one subplot per spot positino
- colour -> energy
- marker -> GA
- dashed lines -> ref tol

In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib.colors import to_hex
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FWHM_FACTOR = 2.3548200450309493


def get_hls_palette_hex(n_colors):
    return [to_hex(c) for c in sns.color_palette("hls", n_colors=n_colors)]


def plotly_fwhm_spot_matrix(
    df,
    gantry,
    device,
    energy=None,              # None / single value / list
    gantry_angle=None,        # None / single value / list
    n_months=12,
    fwhm_col="hor_fwhm",      # "hor_fwhm" / "vert_fwhm" / "bltr_fwhm" / "tlbr_fwhm" / "ave_fwhm"
    ref_df=None,
    ref_source="TPS",
    tol_frac=0.10
):
    
    # Restrict the plotted data to the requested recent time window
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    # Measured FWHM columns available in the input data.
    base_cols = ["hor_fwhm", "vert_fwhm", "bltr_fwhm", "tlbr_fwhm"]
    allowed_single = base_cols + ["ave_fwhm"]

    if fwhm_col not in allowed_single:
        raise ValueError(f"fwhm_col must be one of {allowed_single}")

    # Select measurements for the requested machine, device, and date range.
    sel = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date)
    ].copy()

     # Allow energy to be passed as either a single value or a list of values.
    if energy is not None:
        if not isinstance(energy, (list, tuple, set)):
            energy = [energy]
        sel = sel[sel["Energy"].isin(energy)].copy()

    # Allow gantry angle to be passed as either a single value or a list of values.
    if gantry_angle is not None:
        if not isinstance(gantry_angle, (list, tuple, set)):
            gantry_angle = [gantry_angle]
        sel = sel[sel["Gantry Angle"].isin(gantry_angle)].copy()

    if sel.empty:
        print("No data after filtering.")
        return None

    # Standardise spot labels before matching them to the subplot grid.
    sel["Spot"] = sel["Spot"].astype(str).str.strip()

    # Average measured FWHM across the four measured directions
    sel["ave_fwhm"] = sel[base_cols].mean(axis=1)

    # Identify the plotted energy and gantry-angle groups after filtering.
    energies = sorted(sel["Energy"].dropna().unique())
    gantry_angles = sorted(sel["Gantry Angle"].dropna().unique())

    # Assign one colour per energy.
    energy_colors = dict(zip(energies, get_hls_palette_hex(len(energies))))

    # Assign marker symbols to the 4 gantry angles
    ga_symbols = {
        0: "circle",
        90: "square",
        180: "diamond",
        270: "x",
    }

     # Define the physical spot layout for 3000 and 4000
    if str(device).strip().upper() == "XRV-3000":
        spot_grid = [
            ["Top-Left", "Top-Centre", "Top-Right"],
            ["Left", "Centre", "Right"],
            ["Bottom-Left", "Bottom-Centre", "Bottom-Right"]
        ]
    else:
        spot_grid = [
            ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right"],
            ["Top-Left", "Top-Centre", "Top-Right"],
            ["Left", "Centre", "Right"],
            ["Bottom-Left", "Bottom-Centre", "Bottom-Right"],
            ["Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]
        ]

    n_rows = len(spot_grid)
    n_cols = len(spot_grid[0])

    spot_order = [s for row in spot_grid for s in row] # Flatten the spot layout for subplot titles and iteration

    spot_pos = {
        spot_grid[r][c]: (r + 1, c + 1) # Map each spot label to its subplot row and column
        for r in range(n_rows)
        for c in range(n_cols)
    }

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=spot_order,
        shared_xaxes=True,
        shared_yaxes=True,
        vertical_spacing=0.06 if n_rows == 3 else 0.04,
        horizontal_spacing=0.025
    )

    legend_done = set() # Track which energy/GA combinations have already been added to the legend.
    trace_meta = [] # Store trace metadata so the dropdown controls can update trace visibility.

    x_min = sel["ADate"].min()
    x_max = sel["ADate"].max()

    for e in energies:  # Select data for one energy and one gantry angle at a time to plot them as separate traces.
        for ga in gantry_angles:   
            sub = sel[
                (sel["Energy"] == e) &
                (sel["Gantry Angle"] == ga)
            ].copy()

            if sub.empty:
                continue

            color = energy_colors[e]
            symbol = ga_symbols[ga]
            combo_key = (e, ga)
            legend_group = f"E{e}_GA{ga}"
            legend_name = f"{e} | {ga}°"

            # Determine the reference FWHM value used for tolerance lines.
            tol_center = None

            if ref_df is not None:
                # TPS reference data are stored in ref_df as source G1 at GA 90.
                if str(ref_source).strip().upper() == "TPS":
                    ref_src = "G1"
                    ref_ga = 90
                else:
                    ref_src = ref_source
                    ref_ga = ga

                ref = ref_df[
                    (
                        ref_df["source"].astype(str).str.strip().str.upper()
                        == str(ref_src).strip().upper()
                    ) &
                    (
                        pd.to_numeric(ref_df["gantry"], errors="coerce")
                        == ref_ga
                    ) &
                    (
                        pd.to_numeric(ref_df["energy"], errors="coerce")
                        == e
                    )
                ]

                if not ref.empty:
                    r = ref.iloc[0]

                    x_sigma = float(r["x_stddev"])
                    y_sigma = float(r["y_stddev"])

                    # Convert sigma to FWHM for horizontal and vertical directions.
                    hor_center = x_sigma * FWHM_FACTOR
                    vert_center = y_sigma * FWHM_FACTOR

                    # For diagonal and average FWHM, use the mean of horizontal and vertical reference FWHM.
                    ave_center = (hor_center + vert_center) / 2.0

                    if fwhm_col == "hor_fwhm":
                        tol_center = hor_center
                    elif fwhm_col == "vert_fwhm":
                        tol_center = vert_center
                    else:
                        tol_center = ave_center

            for spot in spot_order:
                row, col = spot_pos[spot]
                sub_spot = sub[sub["Spot"] == spot].sort_values("ADate")

                if tol_center is not None:
                    y_hi = tol_center * (1 + tol_frac)
                    y_lo = tol_center * (1 - tol_frac)

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_hi, y_hi],
                            mode="lines",
                            line=dict(color=color, width=1, dash="dash"),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip"
                        ),
                        row=row,
                        col=col
                    )

                    trace_meta.append(
                        {"energy": e, "ga": ga, "kind": "tol"}
                    )

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_lo, y_lo],
                            mode="lines",
                            line=dict(color=color, width=1, dash="dash"),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip"
                        ),
                        row=row,
                        col=col
                    )

                    trace_meta.append(
                        {"energy": e, "ga": ga, "kind": "tol"}
                    )

                if sub_spot.empty: # If no measurement exists for this spot, leave only the tolerance lines
                    continue

                showlegend = combo_key not in legend_done # Show each energy/GA combination once in the legend.

                fig.add_trace(
                    go.Scatter(
                        x=sub_spot["ADate"],
                        y=sub_spot[fwhm_col],
                        mode="lines+markers",
                        name=legend_name,
                        legendgroup=legend_group,
                        showlegend=showlegend,
                        line=dict(color=color, width=1),
                        marker=dict(
                            symbol=symbol,
                            size=12,
                            color=color,
                            opacity=0.65,
                            line=dict(width=2)
                        ),
                        hovertemplate=(
                            f"Spot={spot}<br>"
                            f"Energy={e} MeV<br>"
                            f"GA={ga}<br>"
                            "Date=%{x|%Y-%m-%d}<br>"
                            f"{fwhm_col}=%{{y:.2f}}<extra></extra>"
                        )
                    ),
                    row=row,
                    col=col
                )

                trace_meta.append(
                    {"energy": e, "ga": ga, "kind": "data"}
                )

                legend_done.add(combo_key)

    # Visibility helpers for the Plotly buttons and dropdown menu.

    def visible_all():
        return [True] * len(trace_meta)

    def visible_none():
        return ["legendonly"] * len(trace_meta)

    def visible_for_ga(target_ga):
        return [
            True if m["ga"] == target_ga else "legendonly"
            for m in trace_meta
        ]

    def visible_for_energy(target_energy):
        return [
            True if m["energy"] == target_energy else "legendonly"
            for m in trace_meta
        ]

    # Dropdown presets for common viewing filters.
    
    preset_buttons = [
        dict(
            label="All",
            method="update",
            args=[{"visible": visible_all()}]
        ),
        dict(
            label="None",
            method="update",
            args=[{"visible": visible_none()}]
        ),
    ]

    for ga in gantry_angles:
        preset_buttons.append(
            dict(
                label=f"GA {ga} only",
                method="update",
                args=[{"visible": visible_for_ga(ga)}]
            )
        )

    for e in energies:
        preset_buttons.append(
            dict(
                label=f"{e} MeV only",
                method="update",
                args=[{"visible": visible_for_energy(e)}]
            )
        )

    fig.update_layout(
        height=900 if n_rows == 3 else 1200,
        width=1650 if n_rows == 3 else 1850,
        margin=dict(l=45, r=290, t=210, b=80),
        template="plotly_white",
        title=dict(
            text=f"{gantry} {device} — {fwhm_col} by spot location",
            x=0.02,
            y=0.98,
            xanchor="left"
        ),
        legend_title_text="Energy | GA",
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=1.02,
            groupclick="togglegroup",
            font=dict(size=11)
        ),
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.00,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=False,
                buttons=[
                    dict(
                        label="Show all",
                        method="update",
                        args=[{"visible": visible_all()}]
                    ),
                    dict(
                        label="Hide all",
                        method="update",
                        args=[{"visible": visible_none()}]
                    ),
                ],
            ),
            dict(
                type="dropdown",
                direction="down",
                x=0.23,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=preset_buttons
            ),
        ],
        annotations=[
            dict(
                text="Quick controls:",
                x=0.00,
                y=1.155,
                xref="paper",
                yref="paper",
                showarrow=False,
                xanchor="left",
                font=dict(size=12)
            ),
            dict(
                text="Preset filter",
                x=0.23,
                y=1.155,
                xref="paper",
                yref="paper",
                showarrow=False,
                xanchor="left",
                font=dict(size=12)
            )
        ]
    )

    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=True)

    fig.show()
    return fig

In [35]:
plotly_fwhm_spot_matrix(
    fwhm_df,
    gantry="Gantry 1",
    device="XRV-3000",
    energy=[70, 100, 150, 200, 240],
    gantry_angle=[0, 90, 180, 270],
    n_months=12,
    fwhm_col="ave_fwhm",
    ref_df=ref_df,
    ref_source="TPS"
)

## Attempt with Dash

In [36]:
from pathlib import Path

import pandas as pd
import seaborn as sns
from matplotlib.colors import to_hex

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dash import Dash, dcc, html, Input, Output


# =============================================================================
# Data paths
# =============================================================================

PROJECT_ROOT = Path.cwd()

FWHM_DATA_PATH = PROJECT_ROOT / "data" / "xlsx_exported_from_access" / "SpotPositionResults.xlsx"

REF_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "xlsx_exported_from_access"
    / "ref"
    / "Ref_RS0_Dist0.xlsx"
)


# =============================================================================
# Plotting function
# =============================================================================

FWHM_FACTOR = 2.3548200450309493


def get_hls_palette_hex(n_colors):
    """
    Return seaborn HLS colours as hex strings for Plotly.
    """
    return [to_hex(c) for c in sns.color_palette("hls", n_colors=n_colors)]


def plotly_fwhm_spot_matrix(
    df,
    gantry,
    device,
    energy=None,
    gantry_angle=None,
    n_months=12,
    fwhm_col="hor_fwhm",
    ref_df=None,
    ref_source="TPS",
    tol_frac=0.10,
    show=False
):
    """
    Plot FWHM measurements by spot position.

    The output is a Plotly subplot matrix:
    - one subplot per spot location
    - colour represents energy
    - marker symbol represents gantry angle
    - optional dashed lines represent reference tolerance limits
    """

    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    base_cols = ["hor_fwhm", "vert_fwhm", "bltr_fwhm", "tlbr_fwhm"]
    allowed_single = base_cols + ["ave_fwhm"]

    if fwhm_col not in allowed_single:
        raise ValueError(f"fwhm_col must be one of {allowed_single}")

    df = df.copy()
    df["ADate"] = pd.to_datetime(df["ADate"], errors="coerce")

    sel = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date)
    ].copy()

    if energy is not None:
        if not isinstance(energy, (list, tuple, set)):
            energy = [energy]
        sel = sel[sel["Energy"].isin(energy)].copy()

    if gantry_angle is not None:
        if not isinstance(gantry_angle, (list, tuple, set)):
            gantry_angle = [gantry_angle]
        sel = sel[sel["Gantry Angle"].isin(gantry_angle)].copy()

    if sel.empty:
        return None

    sel["Spot"] = sel["Spot"].astype(str).str.strip()

    # Average measured FWHM across the four measured directions.
    sel["ave_fwhm"] = sel[base_cols].mean(axis=1)

    energies = sorted(sel["Energy"].dropna().unique())
    gantry_angles = sorted(sel["Gantry Angle"].dropna().unique())

    energy_colors = dict(zip(energies, get_hls_palette_hex(len(energies))))

    # The measurement uses these four gantry angles.
    ga_symbols = {
        0: "circle",
        90: "square",
        180: "diamond",
        270: "x",
    }

    # Define the spot layout for each device type.
    if str(device).strip().upper() == "XRV-3000":
        spot_grid = [
            ["Top-Left", "Top-Centre", "Top-Right"],
            ["Left", "Centre", "Right"],
            ["Bottom-Left", "Bottom-Centre", "Bottom-Right"],
        ]
    else:
        spot_grid = [
            ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right"],
            ["Top-Left", "Top-Centre", "Top-Right"],
            ["Left", "Centre", "Right"],
            ["Bottom-Left", "Bottom-Centre", "Bottom-Right"],
            ["Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"],
        ]

    n_rows = len(spot_grid)
    n_cols = len(spot_grid[0])

    spot_order = [s for row in spot_grid for s in row]

    spot_pos = {
        spot_grid[r][c]: (r + 1, c + 1)
        for r in range(n_rows)
        for c in range(n_cols)
    }

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=spot_order,
        shared_xaxes=True,
        shared_yaxes=True,
        vertical_spacing=0.06 if n_rows == 3 else 0.04,
        horizontal_spacing=0.025,
    )

    legend_done = set()

    x_min = sel["ADate"].min()
    x_max = sel["ADate"].max()

    for e in energies:
        for ga in gantry_angles:
            sub = sel[
                (sel["Energy"] == e) &
                (sel["Gantry Angle"] == ga)
            ].copy()

            if sub.empty:
                continue

            color = energy_colors[e]
            symbol = ga_symbols[ga]
            combo_key = (e, ga)
            legend_group = f"E{e}_GA{ga}"
            legend_name = f"{e} | {ga}°"

            # Determine the reference FWHM value used for tolerance lines.
            tol_center = None

            if ref_df is not None:
                # TPS reference data are stored in ref_df as source G1 at GA 90.
                if str(ref_source).strip().upper() == "TPS":
                    ref_src = "G1"
                    ref_ga = 90
                else:
                    ref_src = ref_source
                    ref_ga = ga

                ref = ref_df[
                    (
                        ref_df["source"].astype(str).str.strip().str.upper()
                        == str(ref_src).strip().upper()
                    ) &
                    (
                        pd.to_numeric(ref_df["gantry"], errors="coerce")
                        == ref_ga
                    ) &
                    (
                        pd.to_numeric(ref_df["energy"], errors="coerce")
                        == e
                    )
                ]

                if not ref.empty:
                    r = ref.iloc[0]

                    x_sigma = float(r["x_stddev"])
                    y_sigma = float(r["y_stddev"])

                    hor_center = x_sigma * FWHM_FACTOR
                    vert_center = y_sigma * FWHM_FACTOR

                    # For diagonal and average FWHM, use the mean of horizontal and vertical reference FWHM.
                    ave_center = (hor_center + vert_center) / 2.0

                    if fwhm_col == "hor_fwhm":
                        tol_center = hor_center
                    elif fwhm_col == "vert_fwhm":
                        tol_center = vert_center
                    else:
                        tol_center = ave_center

            for spot in spot_order:
                row, col = spot_pos[spot]
                sub_spot = sub[sub["Spot"] == spot].sort_values("ADate")

                # Keep tolerance lines visible even if the spot has no measured data.
                if tol_center is not None:
                    y_hi = tol_center * (1 + tol_frac)
                    y_lo = tol_center * (1 - tol_frac)

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_hi, y_hi],
                            mode="lines",
                            line=dict(color=color, width=1, dash="dash"),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip",
                        ),
                        row=row,
                        col=col,
                    )

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_lo, y_lo],
                            mode="lines",
                            line=dict(color=color, width=1, dash="dash"),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip",
                        ),
                        row=row,
                        col=col,
                    )

                if sub_spot.empty:
                    continue

                showlegend = combo_key not in legend_done

                fig.add_trace(
                    go.Scatter(
                        x=sub_spot["ADate"],
                        y=sub_spot[fwhm_col],
                        mode="lines+markers",
                        name=legend_name,
                        legendgroup=legend_group,
                        showlegend=showlegend,
                        line=dict(color=color, width=1),
                        marker=dict(
                            symbol=symbol,
                            size=12,
                            color=color,
                            opacity=0.65,
                            line=dict(width=2),
                        ),
                        hovertemplate=(
                            f"Spot={spot}<br>"
                            f"Energy={e} MeV<br>"
                            f"GA={ga}<br>"
                            "Date=%{x|%Y-%m-%d}<br>"
                            f"{fwhm_col}=%{{y:.2f}}<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )

                legend_done.add(combo_key)

    # No fixed width here. Dash controls the figure width through dcc.Graph.
    fig.update_layout(
        autosize=True,
        height=820 if n_rows == 3 else 1080,
        margin=dict(l=35, r=205, t=95, b=55),
        template="plotly_white",
        title=dict(
            text=f"{gantry} {device} — {fwhm_col} by spot location",
            x=0.02,
            y=0.98,
            xanchor="left",
        ),
        legend_title_text="Energy | GA",
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=1.02,
            groupclick="togglegroup",
            font=dict(size=10),
        ),
    )

    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=True)

    if show:
        fig.show()

    return fig


# =============================================================================
# Data loading
# =============================================================================

def load_fwhm_data():
    """
    Use fwhm_df from the notebook if it already exists.
    Otherwise, load from the Excel file path above.
    """
    global fwhm_df

    try:
        fwhm_df
        df = fwhm_df.copy()
    except NameError:
        df = pd.read_excel(FWHM_DATA_PATH)

    df["ADate"] = pd.to_datetime(df["ADate"], errors="coerce")
    df["MachineName"] = df["MachineName"].astype(str).str.strip()
    df["Device"] = df["Device"].astype(str).str.strip()
    df["Energy"] = pd.to_numeric(df["Energy"], errors="coerce").astype("Int64")
    df["Gantry Angle"] = pd.to_numeric(df["Gantry Angle"], errors="coerce").astype("Int64")

    return df


def load_reference_data():
    """
    Use ref_df from the notebook if it already exists.
    Otherwise, load from REF_DATA_PATH if available.
    """
    global ref_df

    try:
        ref_df
        return ref_df.copy()
    except NameError:
        if REF_DATA_PATH.exists():
            return pd.read_excel(REF_DATA_PATH)
        return None


fwhm_df = load_fwhm_data()
ref_df = load_reference_data()


# =============================================================================
# Dash helpers
# =============================================================================

def make_options(values):
    """
    Convert values into Dash dropdown options.
    """
    clean_values = []

    for value in values:
        if pd.isna(value):
            continue

        if hasattr(value, "item"):
            value = value.item()

        clean_values.append(value)

    return [{"label": str(v), "value": v} for v in clean_values]


def blank_figure(message):
    """
    Return an empty Plotly figure with a central message.
    """
    fig = go.Figure()

    fig.update_layout(
        template="plotly_white",
        xaxis={"visible": False},
        yaxis={"visible": False},
        annotations=[
            {
                "text": message,
                "xref": "paper",
                "yref": "paper",
                "x": 0.5,
                "y": 0.5,
                "showarrow": False,
                "font": {"size": 18},
            }
        ],
        height=700,
    )

    return fig


machine_options = make_options(sorted(fwhm_df["MachineName"].dropna().unique()))
device_options = make_options(sorted(fwhm_df["Device"].dropna().unique()))
energy_options = make_options(sorted(fwhm_df["Energy"].dropna().unique()))
gantry_angle_options = make_options(sorted(fwhm_df["Gantry Angle"].dropna().unique()))

fwhm_options = [
    {"label": "Average FWHM", "value": "ave_fwhm"},
    {"label": "Horizontal FWHM", "value": "hor_fwhm"},
    {"label": "Vertical FWHM", "value": "vert_fwhm"},
    {"label": "BLTR diagonal FWHM", "value": "bltr_fwhm"},
    {"label": "TLBR diagonal FWHM", "value": "tlbr_fwhm"},
]


# =============================================================================
# Dash app
# =============================================================================

app = Dash(__name__)
server = app.server


APP_STYLE_OPEN = {
    "display": "grid",
    "gridTemplateColumns": "280px minmax(0, 1fr)",
    "minHeight": "100vh",
    "fontFamily": "Arial, sans-serif",
    "overflowX": "hidden",
}

APP_STYLE_COLLAPSED = {
    "display": "grid",
    "gridTemplateColumns": "54px minmax(0, 1fr)",
    "minHeight": "100vh",
    "fontFamily": "Arial, sans-serif",
    "overflowX": "hidden",
}

SIDEBAR_STYLE_OPEN = {
    "width": "100%",
    "boxSizing": "border-box",
    "padding": "16px",
    "backgroundColor": "#111824",
    "color": "#edf3fb",
    "height": "100vh",
    "overflowY": "auto",
    "overflowX": "hidden",
    "borderRight": "1px solid #26354b",
}

SIDEBAR_STYLE_COLLAPSED = {
    "width": "100%",
    "boxSizing": "border-box",
    "padding": "10px 6px",
    "backgroundColor": "#111824",
    "color": "#edf3fb",
    "height": "100vh",
    "overflowY": "hidden",
    "overflowX": "hidden",
    "borderRight": "1px solid #26354b",
}

MAIN_STYLE = {
    "padding": "22px",
    "backgroundColor": "#f7f9fc",
    "minHeight": "100vh",
    "minWidth": "0",
    "overflowX": "hidden",
}

LABEL_STYLE = {
    "fontWeight": "600",
    "fontSize": "12px",
    "marginTop": "13px",
    "marginBottom": "6px",
    "display": "block",
}

DROPDOWN_STYLE = {
    "width": "100%",
    "minWidth": "0",
    "fontSize": "12px",
    "color": "#111",
}

INPUT_STYLE = {
    "width": "100%",
    "boxSizing": "border-box",
    "padding": "8px",
    "borderRadius": "8px",
    "border": "1px solid #34465f",
    "fontSize": "12px",
}

PLOT_CARD_STYLE = {
    "backgroundColor": "white",
    "border": "1px solid #d8dee9",
    "borderRadius": "12px",
    "padding": "14px",
    "overflow": "hidden",
    "minWidth": "0",
    "width": "100%",
    "maxWidth": "1450px",
    "boxSizing": "border-box",
}


app.layout = html.Div(
    id="app-shell",
    style=APP_STYLE_OPEN,
    children=[
        html.Aside(
            id="sidebar",
            style=SIDEBAR_STYLE_OPEN,
            children=[
                html.Button(
                    "◀",
                    id="toggle-sidebar",
                    n_clicks=0,
                    title="Hide/show filters",
                    style={
                        "width": "34px",
                        "height": "34px",
                        "border": "1px solid #34465f",
                        "borderRadius": "9px",
                        "backgroundColor": "#0d131d",
                        "color": "#edf3fb",
                        "cursor": "pointer",
                        "fontSize": "15px",
                        "marginBottom": "12px",
                    },
                ),

                html.Div(
                    id="sidebar-content",
                    children=[
                        html.H2(
                            "Spot Position QA",
                            style={
                                "marginTop": "0",
                                "marginBottom": "4px",
                                "fontSize": "20px",
                            },
                        ),
                        html.Div(
                            "MVP dashboard",
                            style={
                                "color": "#9dadc2",
                                "fontSize": "13px",
                            },
                        ),

                        html.Hr(
                            style={
                                "borderColor": "#26354b",
                                "margin": "18px 0",
                            }
                        ),

                        html.Div(
                            "Filters",
                            style={
                                "fontSize": "12px",
                                "textTransform": "uppercase",
                                "color": "#9dadc2",
                                "fontWeight": "700",
                                "letterSpacing": "0.08em",
                            },
                        ),

                        html.Label("Gantry", style=LABEL_STYLE),
                        dcc.Dropdown(
                            id="gantry-dropdown",
                            options=machine_options,
                            value=(
                                "Gantry 1"
                                if "Gantry 1" in fwhm_df["MachineName"].unique()
                                else machine_options[0]["value"]
                            ),
                            clearable=False,
                            style=DROPDOWN_STYLE,
                        ),

                        html.Label("Device", style=LABEL_STYLE),
                        dcc.Dropdown(
                            id="device-dropdown",
                            options=device_options,
                            value=(
                                "XRV-3000"
                                if "XRV-3000" in fwhm_df["Device"].unique()
                                else device_options[0]["value"]
                            ),
                            clearable=False,
                            style=DROPDOWN_STYLE,
                        ),

                        html.Label("Energy", style=LABEL_STYLE),
                        dcc.Dropdown(
                            id="energy-dropdown",
                            options=energy_options,
                            value=[70, 100, 150, 200, 240],
                            multi=True,
                            clearable=False,
                            style=DROPDOWN_STYLE,
                        ),

                        html.Label("Gantry angle", style=LABEL_STYLE),
                        dcc.Dropdown(
                            id="gantry-angle-dropdown",
                            options=gantry_angle_options,
                            value=[0, 90, 180, 270],
                            multi=True,
                            clearable=False,
                            style=DROPDOWN_STYLE,
                        ),

                        html.Label("FWHM value", style=LABEL_STYLE),
                        dcc.Dropdown(
                            id="fwhm-column-dropdown",
                            options=fwhm_options,
                            value="ave_fwhm",
                            clearable=False,
                            style=DROPDOWN_STYLE,
                        ),

                        html.Label("Months back", style=LABEL_STYLE),
                        dcc.Input(
                            id="months-input",
                            type="number",
                            value=12,
                            min=1,
                            max=60,
                            step=1,
                            style=INPUT_STYLE,
                        ),

                        html.Label("Reference source", style=LABEL_STYLE),
                        dcc.Dropdown(
                            id="ref-source-dropdown",
                            options=[
                                {"label": "TPS", "value": "TPS"},
                            ],
                            value="TPS",
                            clearable=False,
                            style=DROPDOWN_STYLE,
                        ),

                        html.Div(
                            "The graph updates automatically when a filter changes.",
                            style={
                                "color": "#9dadc2",
                                "fontSize": "12px",
                                "marginTop": "16px",
                                "lineHeight": "1.4",
                            },
                        ),
                    ],
                ),
            ],
        ),

        html.Main(
            style=MAIN_STYLE,
            children=[
                html.Div(
                    style={
                        "display": "flex",
                        "justifyContent": "space-between",
                        "alignItems": "baseline",
                        "marginBottom": "14px",
                        "maxWidth": "1450px",
                    },
                    children=[
                        html.Div(
                            children=[
                                html.H1(
                                    "Spot size QA",
                                    style={
                                        "margin": "0",
                                        "fontSize": "28px",
                                    },
                                ),
                                html.Div(
                                    "FWHM by spot location, energy and gantry angle.",
                                    style={
                                        "color": "#5c6777",
                                        "fontSize": "14px",
                                    },
                                ),
                            ],
                        ),
                    ],
                ),

                dcc.Tabs(
                    id="right-tabs",
                    value="spot-size",
                    style={
                        "maxWidth": "1450px",
                    },
                    children=[
                        dcc.Tab(
                            label="Spot size",
                            value="spot-size",
                            children=[
                                html.Div(
                                    style=PLOT_CARD_STYLE,
                                    children=[
                                        dcc.Loading(
                                            type="circle",
                                            children=dcc.Graph(
                                                id="spot-size-graph",
                                                config={
                                                    "displaylogo": False,
                                                    "responsive": True,
                                                },
                                                responsive=True,
                                                style={
                                                    "height": "78vh",
                                                    "width": "100%",
                                                    "minWidth": "0",
                                                },
                                            ),
                                        )
                                    ],
                                )
                            ],
                        )
                    ],
                ),
            ],
        ),
    ],
)


# =============================================================================
# Sidebar collapse callback
# =============================================================================

@app.callback(
    Output("app-shell", "style"),
    Output("sidebar", "style"),
    Output("sidebar-content", "style"),
    Output("toggle-sidebar", "children"),
    Input("toggle-sidebar", "n_clicks"),
)
def toggle_sidebar(n_clicks):
    """
    Collapse or expand the left filter panel.
    """

    collapsed = bool(n_clicks and n_clicks % 2 == 1)

    if collapsed:
        return (
            APP_STYLE_COLLAPSED,
            SIDEBAR_STYLE_COLLAPSED,
            {"display": "none"},
            "▶",
        )

    return (
        APP_STYLE_OPEN,
        SIDEBAR_STYLE_OPEN,
        {"display": "block"},
        "◀",
    )


# =============================================================================
# Plot update callback
# =============================================================================

@app.callback(
    Output("spot-size-graph", "figure"),
    Input("gantry-dropdown", "value"),
    Input("device-dropdown", "value"),
    Input("energy-dropdown", "value"),
    Input("gantry-angle-dropdown", "value"),
    Input("months-input", "value"),
    Input("fwhm-column-dropdown", "value"),
    Input("ref-source-dropdown", "value"),
)
def update_spot_size_plot(
    gantry,
    device,
    energy,
    gantry_angle,
    n_months,
    fwhm_col,
    ref_source,
):
    """
    Update the spot-size figure from the sidebar filters.
    """

    if not gantry or not device:
        return blank_figure("Select a gantry and device.")

    if not energy:
        return blank_figure("Select at least one energy.")

    if not gantry_angle:
        return blank_figure("Select at least one gantry angle.")

    if n_months is None:
        n_months = 12

    fig = plotly_fwhm_spot_matrix(
        df=fwhm_df,
        gantry=gantry,
        device=device,
        energy=energy,
        gantry_angle=gantry_angle,
        n_months=int(n_months),
        fwhm_col=fwhm_col,
        ref_df=ref_df,
        ref_source=ref_source,
        show=False,
    )

    if fig is None:
        return blank_figure("No data after filtering.")

    # Keep sizing controlled by Dash rather than a fixed Plotly width.
    fig.update_layout(
        autosize=True,
        margin=dict(l=35, r=205, t=95, b=55),
    )

    return fig


# =============================================================================
# Run app
# =============================================================================

app.run(
    debug=False,
    port=8050,
    use_reloader=False,
    jupyter_mode="external",
)

Dash app running on http://127.0.0.1:8050/


# Generalisation

In [49]:
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative


FWHM_FACTOR = 2.3548200450309493


def plotly_spot_time_series(
    df,
    option,
    gantry,
    device,
    energy,
    gantry_angle,
    n_months=12,
    ref_df=None,
    ref_source="TPS",
    tol_frac=0.10
):
    """
    Plot spot time-series data as four orientation subplots.

    Parameters
    ----------
    df : DataFrame

    option : str
        "fwhm" | "spot_symmetry"

    gantry : str
        e.g. "Gantry 1", "Gantry 2"

    device : str
        e.g. "XRV-3000", "XRV-4000"

    energy : int

    gantry_angle : int
        e.g. 0, 90, 180, 270

    n_months : int
        Number of months of data to display.

    ref_df : DataFrame or None
        Reference table used for FWHM tolerance.
        Expected columns:
        source, gantry, energy, x_stddev, y_stddev

    ref_source : str
        e.g. "TPS", "G1", "G2"

    tol_frac : float
        Fractional FWHM tolerance.
        e.g. 0.10 = ±10%

    Returns
    -------
    fig
        Plotly figure.
    """

    # ------------------------------------------------------------------
    # Filter data
    # ------------------------------------------------------------------

    start_date = (
        pd.Timestamp.today()
        - pd.DateOffset(months=n_months)
    )

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["Energy"] == energy) &
        (df["Gantry Angle"] == gantry_angle) &
        (df["ADate"] >= start_date)
    ].copy()

    if selected_df.empty:
        print("No data after filtering.")
        return None

    # ------------------------------------------------------------------
    # Define data according to option
    # ------------------------------------------------------------------

    if option == "fwhm":

        plot_cols = {
            "Horizontal": "hor_fwhm",
            "Vertical": "vert_fwhm",
            "BLTR": "bltr_fwhm",
            "TLBR": "tlbr_fwhm"
        }

        y_axis_name = "FWHM (mm)"
        plot_title = "FWHM"

    elif option == "spot_symmetry":

        required_cols = [
            "hor_rt_gradient",
            "hor_lt_gradient",
            "vert_rt_gradient",
            "vert_lt_gradient",
            "bltr_rt_gradient",
            "bltr_lt_gradient",
            "tlbr_rt_gradient",
            "tlbr_lt_gradient"
        ]

        missing_cols = [
            col
            for col in required_cols
            if col not in selected_df.columns
        ]

        if missing_cols:
            raise ValueError(
                "Missing columns required for spot symmetry: "
                f"{missing_cols}"
            )

        # Gradient ratios used for spot symmetry
        selected_df["gr_hor"] = (
            selected_df["hor_rt_gradient"]
            / selected_df["hor_lt_gradient"]
        )

        selected_df["gr_vert"] = (
            selected_df["vert_rt_gradient"]
            / selected_df["vert_lt_gradient"]
        )

        selected_df["gr_bltr"] = (
            selected_df["bltr_rt_gradient"]
            / selected_df["bltr_lt_gradient"]
        )

        selected_df["gr_tlbr"] = (
            selected_df["tlbr_rt_gradient"]
            / selected_df["tlbr_lt_gradient"]
        )

        plot_cols = {
            "Horizontal": "gr_hor",
            "Vertical": "gr_vert",
            "BLTR": "gr_bltr",
            "TLBR": "gr_tlbr"
        }

        y_axis_name = "Gradient Ratio (RT/LT)"
        plot_title = "Spot Symmetry"

    else:
        raise ValueError(
            "option must be 'fwhm' or 'spot_symmetry'"
        )

    # ------------------------------------------------------------------
    # Check plotting columns exist
    # ------------------------------------------------------------------

    missing_plot_cols = [
        col
        for col in plot_cols.values()
        if col not in selected_df.columns
    ]

    if missing_plot_cols:
        raise ValueError(
            f"Missing plotting columns: {missing_plot_cols}"
        )

    # ------------------------------------------------------------------
    # Colours and symbols
    # ------------------------------------------------------------------

    spots = (
        selected_df["Spot"]
        .dropna()
        .unique()
        .tolist()
    )

    # Standard Plotly discrete colour palette
    palette = qualitative.Plotly

    spot_colours = {
        spot: palette[i % len(palette)]
        for i, spot in enumerate(spots)
    }

    symbols = [
        "circle",
        "square",
        "diamond",
        "cross",
        "x",
        "triangle-up",
        "triangle-down",
        "triangle-left",
        "triangle-right",
        "pentagon",
        "hexagon",
        "star"
    ]

    spot_symbols = {
        spot: symbols[i % len(symbols)]
        for i, spot in enumerate(spots)
    }

    # ------------------------------------------------------------------
    # Create 2 x 2 figure
    # ------------------------------------------------------------------

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=list(plot_cols.keys()),
        shared_xaxes=True,
        horizontal_spacing=0.08,
        vertical_spacing=0.13
    )

    subplot_positions = {
        "Horizontal": (1, 1),
        "Vertical": (1, 2),
        "BLTR": (2, 1),
        "TLBR": (2, 2)
    }

    # ------------------------------------------------------------------
    # Add data
    # ------------------------------------------------------------------

    for orientation, parameter in plot_cols.items():

        row, col = subplot_positions[orientation]

        for spot in spots:

            spot_df = (
                selected_df[
                    selected_df["Spot"] == spot
                ]
                .sort_values("ADate")
            )

            if spot_df.empty:
                continue

            fig.add_trace(
                go.Scatter(
                    x=spot_df["ADate"],
                    y=spot_df[parameter],

                    mode="markers+lines",

                    name=spot,
                    legendgroup=spot,

                    # Only display each spot once in legend
                    showlegend=(orientation == "Horizontal"),

                    marker=dict(
                        size=12,
                        symbol=spot_symbols[spot],
                        color=spot_colours[spot],
                        opacity=0.8,
                        line=dict(width=2)
                    ),

                    line=dict(
                        width=1,
                        color=spot_colours[spot]
                    ),

                    hovertemplate=(
                        "Date: %{x|%Y-%m-%d}<br>"
                        f"{orientation}: %{{y:.3f}}<br>"
                        "Spot: %{fullData.name}"
                        "<extra></extra>"
                    )
                ),
                row=row,
                col=col
            )

    # ------------------------------------------------------------------
    # FWHM reference / tolerance
    # ------------------------------------------------------------------

    if option == "fwhm" and ref_df is not None:

        # TPS reference uses G1 GA90
        if str(ref_source).strip().upper() == "TPS":
            ref_src = "G1"
            ref_ga = 90

        else:
            ref_src = ref_source
            ref_ga = gantry_angle

        ref = ref_df[
            (
                ref_df["source"]
                .astype(str)
                .str.strip()
                .str.upper()
                ==
                str(ref_src)
                .strip()
                .upper()
            )
            &
            (
                pd.to_numeric(
                    ref_df["gantry"],
                    errors="coerce"
                )
                == ref_ga
            )
            &
            (
                pd.to_numeric(
                    ref_df["energy"],
                    errors="coerce"
                )
                == energy
            )
        ]

        if not ref.empty:

            r = ref.iloc[0]

            x_sigma = float(r["x_stddev"])
            y_sigma = float(r["y_stddev"])

            hor_center = (
                x_sigma
                * FWHM_FACTOR
            )

            vert_center = (
                y_sigma
                * FWHM_FACTOR
            )

            diag_center = (
                hor_center
                + vert_center
            ) / 2

            tolerance_centres = {
                "Horizontal": hor_center,
                "Vertical": vert_center,
                "BLTR": diag_center,
                "TLBR": diag_center
            }

            for orientation, center in tolerance_centres.items():

                row, col = subplot_positions[orientation]

                fig.add_hline(
                    y=center * (1 + tol_frac),
                    line_dash="dash",
                    line_color="grey",
                    line_width=2,
                    row=row,
                    col=col
                )

                fig.add_hline(
                    y=center * (1 - tol_frac),
                    line_dash="dash",
                    line_color="grey",
                    line_width=2,
                    row=row,
                    col=col
                )

    # ------------------------------------------------------------------
    # Title
    # ------------------------------------------------------------------

    if option == "fwhm" and ref_df is not None:

        tolerance_text = (
            f"<br><sup>"
            f"Tolerance: {ref_source} FWHM "
            f"±{tol_frac:.0%}"
            f"</sup>"
        )

    else:
        tolerance_text = ""

    title = (
        f"{gantry} - {device} - "
        f"{energy} MeV - GA {gantry_angle}°"
        f" - {plot_title}"
        f"{tolerance_text}"
    )

    # ------------------------------------------------------------------
    # Layout
    # ------------------------------------------------------------------

    fig.update_layout(
        title=title,
        height=700,
        template="plotly_white",

        legend_title_text="Spot",

        legend=dict(
            orientation="v",
            x=1.02,
            xanchor="left",
            y=1,
            yanchor="top"
        ),

        margin=dict(
            l=70,
            r=180,
            t=100,
            b=60
        )
    )

    fig.update_xaxes(
        title_text="Date",
        showgrid=True
    )

    fig.update_yaxes(
        title_text=y_axis_name,
        showgrid=True
    )

    fig.show()

    return fig

In [51]:
fig = plotly_spot_time_series(
    df=df,
    option="fwhm",
    gantry="Gantry 1",
    device="XRV-3000",
    energy=100,
    gantry_angle=0,
    n_months=12,
    ref_df=ref_df,
    ref_source="TPS",
    tol_frac=0.10
)

In [50]:
fig = plotly_spot_time_series(
    df=df,
    option="spot_symmetry",
    gantry="Gantry 1",
    device="XRV-3000",
    energy=100,
    gantry_angle=0,
    n_months=12
)

In [60]:
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative


FWHM_FACTOR = 2.3548200450309493


def plotly_spot_matrix(
    df,
    option,
    parameter,
    gantry,
    device,
    energy=None,              # None / single value / list
    gantry_angle=None,        # None / single value / list
    n_months=12,
    ref_df=None,
    ref_source="TPS",
    tol_frac=0.10
):
    """
    Plot time-series data in the physical spot-position matrix.

    Parameters
    ----------
    df : DataFrame

    option : str
        "fwhm" | "spot_symmetry"

    parameter : str
        "hor" | "vert" | "bltr" | "tlbr"
        "ave" is also available for FWHM.

    gantry : str

    device : str
        "XRV-3000" | "XRV-4000"

    energy :
        None, single value, or list of values.

    gantry_angle :
        None, single value, or list of values.

    n_months : int

    ref_df : DataFrame or None

    ref_source : str

    tol_frac : float
        e.g. 0.10 = ±10%

    Returns
    -------
    fig
        Plotly figure.
    """

    # ------------------------------------------------------------------
    # Restrict data to requested time window
    # ------------------------------------------------------------------

    start_date = (
        pd.Timestamp.today()
        - pd.DateOffset(months=n_months)
    )

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date)
    ].copy()

    # ------------------------------------------------------------------
    # Optional energy filter
    # ------------------------------------------------------------------

    if energy is not None:

        if not isinstance(energy, (list, tuple, set)):
            energy = [energy]

        selected_df = selected_df[
            selected_df["Energy"].isin(energy)
        ].copy()

    # ------------------------------------------------------------------
    # Optional gantry-angle filter
    # ------------------------------------------------------------------

    if gantry_angle is not None:

        if not isinstance(gantry_angle, (list, tuple, set)):
            gantry_angle = [gantry_angle]

        selected_df = selected_df[
            selected_df["Gantry Angle"].isin(gantry_angle)
        ].copy()

    if selected_df.empty:
        print("No data after filtering.")
        return None

    # Standardise spot labels
    selected_df["Spot"] = (
        selected_df["Spot"]
        .astype(str)
        .str.strip()
    )

    # ------------------------------------------------------------------
    # Define data according to option
    # ------------------------------------------------------------------

    if option == "fwhm":

        base_cols = [
            "hor_fwhm",
            "vert_fwhm",
            "bltr_fwhm",
            "tlbr_fwhm"
        ]

        selected_df["ave_fwhm"] = (
            selected_df[base_cols]
            .mean(axis=1)
        )

        parameter_map = {
            "hor": "hor_fwhm",
            "vert": "vert_fwhm",
            "bltr": "bltr_fwhm",
            "tlbr": "tlbr_fwhm",
            "ave": "ave_fwhm"
        }

        parameter_names = {
            "hor": "Horizontal FWHM (mm)",
            "vert": "Vertical FWHM (mm)",
            "bltr": "BLTR FWHM (mm)",
            "tlbr": "TLBR FWHM (mm)",
            "ave": "Average FWHM (mm)"
        }

    elif option == "spot_symmetry":

        required_cols = [
            "hor_rt_gradient",
            "hor_lt_gradient",
            "vert_rt_gradient",
            "vert_lt_gradient",
            "bltr_rt_gradient",
            "bltr_lt_gradient",
            "tlbr_rt_gradient",
            "tlbr_lt_gradient"
        ]

        missing_cols = [
            col
            for col in required_cols
            if col not in selected_df.columns
        ]

        if missing_cols:
            raise ValueError(
                "Missing columns required for spot symmetry: "
                f"{missing_cols}"
            )

        # Calculate gradient ratios
        selected_df["gr_hor"] = (
            selected_df["hor_rt_gradient"]
            / selected_df["hor_lt_gradient"]
        )

        selected_df["gr_vert"] = (
            selected_df["vert_rt_gradient"]
            / selected_df["vert_lt_gradient"]
        )

        selected_df["gr_bltr"] = (
            selected_df["bltr_rt_gradient"]
            / selected_df["bltr_lt_gradient"]
        )

        selected_df["gr_tlbr"] = (
            selected_df["tlbr_rt_gradient"]
            / selected_df["tlbr_lt_gradient"]
        )

        parameter_map = {
            "hor": "gr_hor",
            "vert": "gr_vert",
            "bltr": "gr_bltr",
            "tlbr": "gr_tlbr"
        }

        parameter_names = {
            "hor": "Horizontal Gradient Ratio (RT/LT)",
            "vert": "Vertical Gradient Ratio (RT/LT)",
            "bltr": "BLTR Gradient Ratio (RT/LT)",
            "tlbr": "TLBR Gradient Ratio (RT/LT)"
        }

    else:
        raise ValueError(
            "option must be 'fwhm' or 'spot_symmetry'"
        )

    # ------------------------------------------------------------------
    # Validate selected parameter
    # ------------------------------------------------------------------

    if parameter not in parameter_map:
        raise ValueError(
            f"For option='{option}', parameter must be one of "
            f"{list(parameter_map.keys())}"
        )

    plot_col = parameter_map[parameter]
    y_axis_name = parameter_names[parameter]

    # ------------------------------------------------------------------
    # Energy colours and GA symbols
    # ------------------------------------------------------------------

    energies = sorted(
        selected_df["Energy"]
        .dropna()
        .unique()
    )

    gantry_angles = sorted(
        selected_df["Gantry Angle"]
        .dropna()
        .unique()
    )

    palette = qualitative.Plotly

    energy_colors = {
        e: palette[i % len(palette)]
        for i, e in enumerate(energies)
    }

    ga_symbols = {
        0: "circle",
        90: "square",
        180: "diamond",
        270: "x"
    }

    # ------------------------------------------------------------------
    # Physical spot layout
    # ------------------------------------------------------------------

    if str(device).strip().upper() == "XRV-3000":

        spot_grid = [
            ["Top-Left", "Top-Centre", "Top-Right"],
            ["Left", "Centre", "Right"],
            ["Bottom-Left", "Bottom-Centre", "Bottom-Right"]
        ]

    else:

        spot_grid = [
            ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right"],
            ["Top-Left", "Top-Centre", "Top-Right"],
            ["Left", "Centre", "Right"],
            ["Bottom-Left", "Bottom-Centre", "Bottom-Right"],
            ["Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]
        ]

    n_rows = len(spot_grid)
    n_cols = len(spot_grid[0])

    spot_order = [
        spot
        for row in spot_grid
        for spot in row
    ]

    spot_pos = {
        spot_grid[r][c]: (r + 1, c + 1)
        for r in range(n_rows)
        for c in range(n_cols)
    }

    # ------------------------------------------------------------------
    # Create subplot matrix
    # ------------------------------------------------------------------

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=spot_order,
        shared_xaxes=True,
        shared_yaxes=True,
        vertical_spacing=0.06 if n_rows == 3 else 0.04,
        horizontal_spacing=0.025
    )

    legend_done = set()
    trace_meta = []

    x_min = selected_df["ADate"].min()
    x_max = selected_df["ADate"].max()

    # ------------------------------------------------------------------
    # Add traces
    # ------------------------------------------------------------------

    for e in energies:

        for ga in gantry_angles:

            sub = selected_df[
                (selected_df["Energy"] == e) &
                (selected_df["Gantry Angle"] == ga)
            ].copy()

            if sub.empty:
                continue

            color = energy_colors[e]
            symbol = ga_symbols.get(ga, "circle")

            combo_key = (e, ga)

            legend_group = f"E{e}_GA{ga}"
            legend_name = f"{e} | {ga}°"

            # ----------------------------------------------------------
            # Determine FWHM tolerance reference
            # ----------------------------------------------------------

            tol_center = None

            if option == "fwhm" and ref_df is not None:

                if str(ref_source).strip().upper() == "TPS":

                    ref_src = "G1"
                    ref_ga = 90

                else:

                    ref_src = ref_source
                    ref_ga = ga

                ref = ref_df[
                    (
                        ref_df["source"]
                        .astype(str)
                        .str.strip()
                        .str.upper()
                        ==
                        str(ref_src)
                        .strip()
                        .upper()
                    )
                    &
                    (
                        pd.to_numeric(
                            ref_df["gantry"],
                            errors="coerce"
                        )
                        == ref_ga
                    )
                    &
                    (
                        pd.to_numeric(
                            ref_df["energy"],
                            errors="coerce"
                        )
                        == e
                    )
                ]

                if not ref.empty:

                    r = ref.iloc[0]

                    x_sigma = float(r["x_stddev"])
                    y_sigma = float(r["y_stddev"])

                    hor_center = (
                        x_sigma
                        * FWHM_FACTOR
                    )

                    vert_center = (
                        y_sigma
                        * FWHM_FACTOR
                    )

                    ave_center = (
                        hor_center
                        + vert_center
                    ) / 2.0

                    if parameter == "hor":
                        tol_center = hor_center

                    elif parameter == "vert":
                        tol_center = vert_center

                    else:
                        tol_center = ave_center

            # ----------------------------------------------------------
            # Add each physical spot
            # ----------------------------------------------------------

            for spot in spot_order:

                row, col = spot_pos[spot]

                sub_spot = (
                    sub[sub["Spot"] == spot]
                    .sort_values("ADate")
                )

                # ------------------------------------------------------
                # FWHM tolerance lines
                # ------------------------------------------------------

                if tol_center is not None:

                    y_hi = (
                        tol_center
                        * (1 + tol_frac)
                    )

                    y_lo = (
                        tol_center
                        * (1 - tol_frac)
                    )

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_hi, y_hi],
                            mode="lines",
                            line=dict(
                                color=color,
                                width=1,
                                dash="dash"
                            ),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip"
                        ),
                        row=row,
                        col=col
                    )

                    trace_meta.append(
                        {
                            "energy": e,
                            "ga": ga,
                            "kind": "tol"
                        }
                    )

                    fig.add_trace(
                        go.Scatter(
                            x=[x_min, x_max],
                            y=[y_lo, y_lo],
                            mode="lines",
                            line=dict(
                                color=color,
                                width=1,
                                dash="dash"
                            ),
                            legendgroup=legend_group,
                            showlegend=False,
                            hoverinfo="skip"
                        ),
                        row=row,
                        col=col
                    )

                    trace_meta.append(
                        {
                            "energy": e,
                            "ga": ga,
                            "kind": "tol"
                        }
                    )

                if sub_spot.empty:
                    continue

                showlegend = (
                    combo_key not in legend_done
                )

                fig.add_trace(
                    go.Scatter(
                        x=sub_spot["ADate"],
                        y=sub_spot[plot_col],

                        mode="lines+markers",

                        name=legend_name,
                        legendgroup=legend_group,
                        showlegend=showlegend,

                        line=dict(
                            color=color,
                            width=1
                        ),

                        marker=dict(
                            symbol=symbol,
                            size=12,
                            color=color,
                            opacity=0.65,
                            line=dict(width=2)
                        ),

                        hovertemplate=(
                            f"Spot={spot}<br>"
                            f"Energy={e} MeV<br>"
                            f"GA={ga}°<br>"
                            "Date=%{x|%Y-%m-%d}<br>"
                            f"{y_axis_name}=%{{y:.3f}}"
                            "<extra></extra>"
                        )
                    ),
                    row=row,
                    col=col
                )

                trace_meta.append(
                    {
                        "energy": e,
                        "ga": ga,
                        "kind": "data"
                    }
                )

                legend_done.add(combo_key)

    # ------------------------------------------------------------------
    # Visibility helpers
    # ------------------------------------------------------------------

    def visible_all():
        return [
            True
            for _ in trace_meta
        ]

    def visible_none():
        return [
            "legendonly"
            for _ in trace_meta
        ]

    def visible_for_ga(target_ga):
        return [
            True
            if m["ga"] == target_ga
            else "legendonly"
            for m in trace_meta
        ]

    def visible_for_energy(target_energy):
        return [
            True
            if m["energy"] == target_energy
            else "legendonly"
            for m in trace_meta
        ]

    # ------------------------------------------------------------------
    # Dropdown presets
    # ------------------------------------------------------------------

    preset_buttons = [
        dict(
            label="All",
            method="update",
            args=[
                {
                    "visible": visible_all()
                }
            ]
        ),

        dict(
            label="None",
            method="update",
            args=[
                {
                    "visible": visible_none()
                }
            ]
        )
    ]

    for ga in gantry_angles:

        preset_buttons.append(
            dict(
                label=f"GA {ga} only",
                method="update",
                args=[
                    {
                        "visible": visible_for_ga(ga)
                    }
                ]
            )
        )

    for e in energies:

        preset_buttons.append(
            dict(
                label=f"{e} MeV only",
                method="update",
                args=[
                    {
                        "visible": visible_for_energy(e)
                    }
                ]
            )
        )

    # ------------------------------------------------------------------
    # Layout
    # ------------------------------------------------------------------

    fig.update_layout(

        height=900 if n_rows == 3 else 1200,

        width=1650 if n_rows == 3 else 1850,

        margin=dict(
            l=90,
            r=290,
            t=210,
            b=80
        ),

        template="plotly_white",

        title=dict(
            text=(
                f"{gantry} {device} — "
                f"{y_axis_name} "
                f"by spot location"
            ),
            x=0.02,
            y=0.98,
            xanchor="left"
        ),

        legend_title_text="Energy | GA",

        legend=dict(
            orientation="v",
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=1.02,
            groupclick="togglegroup",
            font=dict(size=11)
        ),

        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.00,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=False,

                buttons=[
                    dict(
                        label="Show all",
                        method="update",
                        args=[
                            {
                                "visible": visible_all()
                            }
                        ]
                    ),

                    dict(
                        label="Hide all",
                        method="update",
                        args=[
                            {
                                "visible": visible_none()
                            }
                        ]
                    )
                ]
            ),

            dict(
                type="dropdown",
                direction="down",
                x=0.23,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=preset_buttons
            )
        ],

        annotations=[
            dict(
                text="Quick controls:",
                x=0.00,
                y=1.155,
                xref="paper",
                yref="paper",
                showarrow=False,
                xanchor="left",
                font=dict(size=12)
            ),

            dict(
                text="Preset filter",
                x=0.23,
                y=1.155,
                xref="paper",
                yref="paper",
                showarrow=False,
                xanchor="left",
                font=dict(size=12)
            )
        ]
    )

    # ------------------------------------------------------------------
    # Shared axes
    # ------------------------------------------------------------------

    fig.update_xaxes(
        showgrid=True
    )

    fig.update_yaxes(
        showgrid=True
    )

    # One shared y-axis label for the whole figure
    fig.add_annotation(
        text=y_axis_name,
        x=-0.045,
        y=0.5,
        xref="paper",
        yref="paper",
        textangle=-90,
        showarrow=False,
        font=dict(size=14)
    )

    return fig

In [62]:
# FWHM
plotly_spot_matrix(
    df=df,
    option="fwhm",
    parameter="hor",
    gantry="Gantry 1",
    device="XRV-3000",
    energy=[100, 150],
    gantry_angle=[0, 90, 180, 270],
    n_months=12,
    ref_df=ref_df,
    ref_source="TPS",
    tol_frac=0.10
)

In [63]:
# Spot symmetry
plotly_spot_matrix(
    df=df,
    option="spot_symmetry",
    parameter="hor",
    gantry="Gantry 1",
    device="XRV-3000",
    energy=[100, 150],
    gantry_angle=[0, 90, 180, 270],
    n_months=12
)